# Multi-Model RT Validation (Colab GPU)

**Goal**: Prove RT is a universal amplifier across models.

**Thesis**: Stronger base model × RT fusion = world-class results.
If RT helps even large models, the argument is bulletproof.

**Models tested** (all fit on T4 16GB):
1. E5-base (110M) — our current champion
2. E5-large-v2 (335M) — bigger E5
3. BGE-large-en-v1.5 (335M) — strong competitor
4. GTE-large-en-v1.5 (434M) — Alibaba's best <1B

**Datasets**: SciFact (fast) + FiQA (hard)

**Target**: RT + large model > InRanker-3B (0.7831) on SciFact

In [ ]:
!pip install -q beir sentence-transformers rank-bm25 numpy pytrec-eval-terrier
!nvidia-smi | head -20

In [ ]:
import json
import os
import re
import time
from collections import defaultdict

import numpy as np
import pytrec_eval
import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', 0)
    print(f"GPU: {name} | VRAM: {vram / 1e9:.1f} GB")

In [ ]:
# ── Models to test ──
# Order: small → large (clear progression)
MODELS = [
    {
        "name": "E5-base",
        "hf_id": "intfloat/e5-base-unsupervised",
        "params": "110M",
        "prefix_q": "query: ",
        "prefix_d": "passage: ",
    },
    {
        "name": "E5-large-v2",
        "hf_id": "intfloat/e5-large-v2",
        "params": "335M",
        "prefix_q": "query: ",
        "prefix_d": "passage: ",
    },
    {
        "name": "BGE-large-en-v1.5",
        "hf_id": "BAAI/bge-large-en-v1.5",
        "params": "335M",
        "prefix_q": "Represent this sentence for searching relevant passages: ",
        "prefix_d": "",
    },
    {
        "name": "GTE-large-en-v1.5",
        "hf_id": "Alibaba-NLP/gte-large-en-v1.5",
        "params": "434M",
        "prefix_q": "",
        "prefix_d": "",
    },
]

# RT fixed params (universal: works across models)
RT_UNIVERSAL = {
    "k_low": 2, "k_high": 5, "top_n": 20,
    "boost_max": 1.2, "score_w": 0.5, "bw": 0.8, "dw": 1.0,
}

# RT E5PT-specific (best on SciFact)
RT_E5PT = {
    "k_low": 3, "k_high": 10, "top_n": 20,
    "boost_max": 1.2, "score_w": 0.5, "bw": 0.8, "dw": 1.4,
}

# RRF baseline
RRF_PARAMS = {"k": 5, "bw": 1.0, "dw": 1.2}

print(f"Testing {len(MODELS)} models")
for m in MODELS:
    print(f"  - {m['name']} ({m['params']})")

In [ ]:
# ── Official evaluation (pytrec_eval) + Fusion functions ──

def evaluate_official(qrels, run_dict, metrics=None):
    """Official BEIR evaluation with pytrec_eval."""
    if metrics is None:
        metrics = {"ndcg_cut_10", "recall_100", "map"}
    qrels_int = {
        qid: {did: int(rel) for did, rel in rels.items()}
        for qid, rels in qrels.items()
    }
    evaluator = pytrec_eval.RelevanceEvaluator(qrels_int, metrics)
    scores = evaluator.evaluate(run_dict)
    result = {}
    for metric in metrics:
        vals = [scores[qid].get(metric, 0) for qid in scores]
        result[metric] = round(sum(vals) / len(vals), 6)
    return result


def simple_rrf(b, d, k=5, bw=1.0, dw=1.2):
    scores = defaultdict(float)
    for rank, (did, _) in enumerate(b):
        scores[did] += bw / (k + rank + 1)
    for rank, (did, _) in enumerate(d):
        scores[did] += dw / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def riverbed_tension(
    b, d, k_low=2, k_high=10, top_n=10,
    boost_max=1.3, score_w=0.3, bw=1.0, dw=1.2,
):
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    agreement = len(b_set & d_set) / len(union) if union else 0.0
    tension = 1.0 - agreement
    adaptive_k = max(1, int(k_low + (k_high - k_low) * tension))
    boost = 1.0 + (boost_max - 1.0) * agreement

    rrf_scores = defaultdict(float)
    presence = defaultdict(int)
    for rank, (did, _) in enumerate(b):
        rrf_scores[did] += bw / (adaptive_k + rank + 1)
        presence[did] += 1
    for rank, (did, _) in enumerate(d):
        rrf_scores[did] += dw / (adaptive_k + rank + 1)
        presence[did] += 1
    for did in rrf_scores:
        if presence[did] >= 2:
            rrf_scores[did] *= boost

    def norm(results):
        if not results:
            return {}
        vals = [s for _, s in results]
        mn, mx = min(vals), max(vals)
        rng = mx - mn if mx > mn else 1.0
        return {did: (s - mn) / rng for did, s in results}

    b_n, d_n = norm(b), norm(d)
    rv = list(rrf_scores.values())
    r_mn, r_mx = min(rv), max(rv)
    r_rng = r_mx - r_mn if r_mx > r_mn else 1.0
    tw = bw + dw

    all_docs = set(rrf_scores) | set(b_n) | set(d_n)
    final = {}
    for did in all_docs:
        r = (rrf_scores.get(did, 0) - r_mn) / r_rng
        s = (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        final[did] = (1 - score_w) * r + score_w * s
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


print("Fusion + official evaluator ready")

In [ ]:
# ── Download datasets ──

DATASETS = {
    "scifact": "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",
    "fiqa": "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip",
}

BASE_DIR = "datasets"
os.makedirs(BASE_DIR, exist_ok=True)

loaded_datasets = {}
for ds_name, url in DATASETS.items():
    data_path = os.path.join(BASE_DIR, ds_name)
    if not os.path.isdir(data_path):
        print(f"Downloading {ds_name}...")
        data_path = util.download_and_unzip(url, BASE_DIR)
    corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")
    loaded_datasets[ds_name] = (corpus, queries, qrels)
    print(f"{ds_name}: {len(corpus)} docs, {len(queries)} queries")

In [ ]:
# ── BM25 indices (one per dataset, model-independent) ──

def tokenize(text):
    return re.findall(r'\w+', text.lower())


def build_bm25(corpus):
    doc_ids = list(corpus.keys())
    tokenized = [
        tokenize(
            f"{corpus[did].get('title', '')} "
            f"{corpus[did].get('text', '')}"
        )
        for did in doc_ids
    ]
    bm25 = BM25Okapi(tokenized)
    return bm25, doc_ids


def search_bm25(bm25, doc_ids, query, top_k=100):
    scores = bm25.get_scores(tokenize(query))
    top_idx = scores.argsort()[-top_k:][::-1]
    return [
        (doc_ids[i], float(scores[i]))
        for i in top_idx if scores[i] > 0
    ]


bm25_indices = {}
for ds_name, (corpus, _, _) in loaded_datasets.items():
    t0 = time.time()
    bm25, doc_ids = build_bm25(corpus)
    bm25_indices[ds_name] = (bm25, doc_ids)
    print(f"{ds_name}: BM25 built in {time.time() - t0:.1f}s")

In [ ]:
# ── Core: encode + evaluate one model on one dataset ──

def encode_corpus(model, corpus, prefix_d, ds_name, model_name):
    """Encode corpus with caching and checkpointing."""
    tag = model_name.replace('/', '_').replace('-', '_')
    cache_path = os.path.join(
        BASE_DIR, f".cache_{ds_name}_{tag}.npz",
    )
    ckpt_path = cache_path + ".ckpt.npz"

    doc_id_list = list(corpus.keys())
    texts = []
    for did in doc_id_list:
        doc = corpus[did]
        title = doc.get('title', '')
        text = doc.get('text', '')
        raw = f"{title} {text}".strip()
        texts.append(f"{prefix_d}{raw}" if prefix_d else raw)

    if os.path.exists(cache_path):
        data = np.load(cache_path)
        print(f"  Cache hit: {cache_path} ({data['embs'].shape})")
        return data["embs"], doc_id_list

    start_idx = 0
    all_embs = []
    if os.path.exists(ckpt_path):
        ckpt = np.load(ckpt_path)
        start_idx = int(ckpt["done"])
        all_embs = [ckpt["embs"]]
        print(f"  Resuming from {start_idx}/{len(texts)}")

    batch_size = 256
    t0 = time.time()
    for i in range(start_idx, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embs = model.encode(
            batch, normalize_embeddings=True,
            show_progress_bar=False, batch_size=128,
        )
        all_embs.append(embs)
        done = min(i + batch_size, len(texts))
        if done % 5000 < batch_size or done == len(texts):
            partial = np.vstack(all_embs)
            np.savez_compressed(ckpt_path, embs=partial, done=done)
            elapsed = time.time() - t0
            speed = (done - start_idx) / elapsed if elapsed > 0 else 0
            eta = (len(texts) - done) / speed if speed > 0 else 0
            print(f"  {done}/{len(texts)} ({elapsed:.0f}s, {speed:.0f} d/s, ETA {eta:.0f}s)")

    passage_embs = np.vstack(all_embs)
    np.savez_compressed(cache_path, embs=passage_embs)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    print(f"  Done: {passage_embs.shape} in {time.time() - t0:.1f}s")
    return passage_embs, doc_id_list


def evaluate_model_dataset(
    model_cfg, ds_name, corpus, queries, qrels,
    bm25, bm25_doc_ids, st_model, passage_embs, doc_id_list,
):
    """Evaluate all fusion strategies for one model+dataset using pytrec_eval."""
    prefix_q = model_cfg["prefix_q"]

    def dense_search(qt, top_k=100):
        q_text = f"{prefix_q}{qt}" if prefix_q else qt
        q = st_model.encode([q_text], normalize_embeddings=True)
        sims = (passage_embs @ q.T).flatten()
        idx = np.argsort(sims)[::-1][:top_k]
        return [(doc_id_list[i], float(sims[i])) for i in idx]

    # Cache queries
    cached = {}
    for qid, qt in queries.items():
        cached[qid] = {
            "bm25": search_bm25(bm25, bm25_doc_ids, qt, top_k=100),
            "dense": dense_search(qt, top_k=100),
        }

    def build_run_dict(fusion_fn, **kw):
        run_dict = {}
        for qid, e in cached.items():
            fused = fusion_fn(e["bm25"], e["dense"], **kw)
            run_dict[qid] = {did: float(s) for did, s in fused[:100]}
        return run_dict

    # Dense only (identity fusion: ignore bm25, return dense)
    dense_run = {}
    for qid, e in cached.items():
        dense_run[qid] = {did: float(s) for did, s in e["dense"][:100]}
    dense_metrics = evaluate_official(qrels, dense_run)

    # Simple RRF
    rrf_run = build_run_dict(simple_rrf, **RRF_PARAMS)
    rrf_metrics = evaluate_official(qrels, rrf_run)

    # RT Universal
    rtu_run = build_run_dict(riverbed_tension, **RT_UNIVERSAL)
    rtu_metrics = evaluate_official(qrels, rtu_run)

    # RT E5PT (originally tuned for E5-PT, test on all)
    rt5_run = build_run_dict(riverbed_tension, **RT_E5PT)
    rt5_metrics = evaluate_official(qrels, rt5_run)

    return {
        "dense": dense_metrics,
        "rrf": rrf_metrics,
        "rt_universal": rtu_metrics,
        "rt_e5pt": rt5_metrics,
    }


print("Core functions ready")

In [ ]:
# ── Main loop: all models × all datasets ──

ALL_RESULTS = {}

for mi, model_cfg in enumerate(MODELS):
    model_name = model_cfg["name"]
    hf_id = model_cfg["hf_id"]
    print(f"\n{'=' * 60}")
    print(f"[{mi+1}/{len(MODELS)}] {model_name} ({model_cfg['params']})")
    print(f"{'=' * 60}")

    # Load model
    print(f"Loading {hf_id}...")
    st_model = SentenceTransformer(hf_id, device=DEVICE)

    ALL_RESULTS[model_name] = {}

    for ds_name, (corpus, queries, qrels) in loaded_datasets.items():
        print(f"\n--- {model_name} on {ds_name} ---")

        # Encode
        passage_embs, doc_id_list = encode_corpus(
            st_model, corpus,
            model_cfg["prefix_d"], ds_name, hf_id,
        )

        # Evaluate
        bm25, bm25_doc_ids = bm25_indices[ds_name]
        results = evaluate_model_dataset(
            model_cfg, ds_name, corpus, queries, qrels,
            bm25, bm25_doc_ids, st_model, passage_embs, doc_id_list,
        )
        ALL_RESULTS[model_name][ds_name] = results

        # Print summary
        d = results["dense"]["ndcg_cut_10"]
        rrf = results["rrf"]["ndcg_cut_10"]
        rtu = results["rt_universal"]["ndcg_cut_10"]
        rt5 = results["rt_e5pt"]["ndcg_cut_10"]
        best_rt = max(rtu, rt5)
        print(f"  Dense={d:.4f} | RRF={rrf:.4f} | RT_U={rtu:.4f} | RT_E5={rt5:.4f}")
        print(f"  Best RT vs Dense: {best_rt - d:+.4f} | vs RRF: {best_rt - rrf:+.4f}")

    # Free GPU memory before loading next model
    del st_model
    torch.cuda.empty_cache()
    print(f"\n[Freed GPU memory for {model_name}]")

print("\n" + "=" * 60)
print("ALL MODELS DONE")
print("=" * 60)

In [ ]:
# ── Summary table ──

print("\n" + "=" * 80)
print("COMPREHENSIVE RESULTS: RT as Universal Amplifier (pytrec_eval official)")
print("=" * 80)

# Reference: InRanker-3B = 0.7831 on SciFact (reranker, 3B params)
INRANKER_SCIFACT = 0.7831

for ds_name in DATASETS:
    print(f"\n--- {ds_name.upper()} (nDCG@10 | R@100 | MAP) ---")
    print(f"{'Model':<22} {'Params':<8} {'Dense':>8} {'RRF':>8} {'RT_U':>8} {'RT_E5':>8} {'Best RT':>8} {'Δ vs Dense':>10} {'Δ vs RRF':>10}")
    print("-" * 100)

    for model_cfg in MODELS:
        mn = model_cfg["name"]
        if mn not in ALL_RESULTS or ds_name not in ALL_RESULTS[mn]:
            continue
        r = ALL_RESULTS[mn][ds_name]
        d = r["dense"]["ndcg_cut_10"]
        rrf = r["rrf"]["ndcg_cut_10"]
        rtu = r["rt_universal"]["ndcg_cut_10"]
        rt5 = r["rt_e5pt"]["ndcg_cut_10"]
        best = max(rtu, rt5)
        print(
            f"{mn:<22} {model_cfg['params']:<8} "
            f"{d:>8.4f} {rrf:>8.4f} {rtu:>8.4f} {rt5:>8.4f} "
            f"{best:>8.4f} {best - d:>+10.4f} {best - rrf:>+10.4f}"
        )

    # Print recall and MAP for best RT variant
    print(f"\n  {'Model':<22} {'Strategy':<14} {'nDCG@10':>8} {'R@100':>8} {'MAP':>8}")
    print(f"  {'-' * 70}")
    for model_cfg in MODELS:
        mn = model_cfg["name"]
        if mn not in ALL_RESULTS or ds_name not in ALL_RESULTS[mn]:
            continue
        r = ALL_RESULTS[mn][ds_name]
        for strat in ["dense", "rrf", "rt_universal", "rt_e5pt"]:
            m = r[strat]
            print(
                f"  {mn:<22} {strat:<14} "
                f"{m['ndcg_cut_10']:>8.4f} {m['recall_100']:>8.4f} {m['map']:>8.4f}"
            )

    if ds_name == "scifact":
        print(f"\nReference: InRanker-3B (reranker) = {INRANKER_SCIFACT}")
        # Check if any model + RT beats InRanker
        for model_cfg in MODELS:
            mn = model_cfg["name"]
            if mn in ALL_RESULTS and ds_name in ALL_RESULTS[mn]:
                r = ALL_RESULTS[mn][ds_name]
                best = max(
                    r["rt_universal"]["ndcg_cut_10"],
                    r["rt_e5pt"]["ndcg_cut_10"],
                )
                if best > INRANKER_SCIFACT:
                    gap = best - INRANKER_SCIFACT
                    print(
                        f"*** {mn} + RT ({best:.4f}) BEATS "
                        f"InRanker-3B ({INRANKER_SCIFACT}) "
                        f"by {gap:+.4f}! ***"
                    )

In [ ]:
# ── Key insight: RT amplification vs model size ──

print("\n" + "=" * 60)
print("RT AMPLIFICATION by Model Size (SciFact)")
print("=" * 60)

if "scifact" in list(ALL_RESULTS.values())[0]:
    rows = []
    for model_cfg in MODELS:
        mn = model_cfg["name"]
        if mn not in ALL_RESULTS:
            continue
        r = ALL_RESULTS[mn].get("scifact", {})
        if not r:
            continue
        d = r["dense"]["ndcg_cut_10"]
        best_rt = max(
            r["rt_universal"]["ndcg_cut_10"],
            r["rt_e5pt"]["ndcg_cut_10"],
        )
        pct = (best_rt - d) / d * 100
        rows.append((mn, model_cfg["params"], d, best_rt, pct))

    print(f"{'Model':<22} {'Params':<8} {'Dense':>8} {'Best RT':>8} {'RT Lift%':>10}")
    print("-" * 60)
    for mn, params, d, best, pct in rows:
        print(f"{mn:<22} {params:<8} {d:>8.4f} {best:>8.4f} {pct:>+9.2f}%")

    print("\nInsight: If RT% stays positive across model sizes,")
    print("RT is a universal amplifier, not a weak-model crutch.")

In [ ]:
# ── Save all results ──

output = {
    "experiment": "multi_model_rt_validation",
    "evaluator": "pytrec_eval 0.5.10 (official BEIR standard)",
    "submission_ready": True,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU",
    "params": {
        "rt_universal": RT_UNIVERSAL,
        "rt_e5pt": RT_E5PT,
        "rrf": RRF_PARAMS,
    },
    "models": [
        {"name": m["name"], "hf_id": m["hf_id"], "params": m["params"]}
        for m in MODELS
    ],
    "datasets": list(DATASETS.keys()),
    "results": ALL_RESULTS,
    "reference": {
        "inranker_3b_scifact": INRANKER_SCIFACT,
        "note": "InRanker-3B uses 3B param cross-encoder reranker",
    },
}

out_file = "multi_model_results.json"
with open(out_file, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved: {out_file}")
print("\n=== COPY THIS JSON ===")
print(json.dumps(output, indent=2))
print("=== END JSON ===")